# Function declarations and library imports

In [ ]:
import datetime
import math
import os
import pathlib
import shutil
import tempfile
import time

import cv2
import matplotlib.pyplot as plt
from matplotlib import patches
from numba import njit,types
from numba.typed import List
import numpy as np
import pandas as pd
from PIL import Image
from pptx import Presentation
import roifile
from scipy.optimize import curve_fit
import seaborn as sns
from skimage.filters import rank
from skimage.morphology import disk
from tifffile import TiffFile

import GUI_HTA as htagui  

In [ ]:
hierarchy = None
contours = None
image0 = None
image1 = None

# List of numpy arrays recording the normal slope, centroid x-coordinate, y-coordinate,
# and the index assigned by findContours, for each cell and each time point
mxy = None
# List of dictionaries recording cell size; index is the contour number at time t
cellsize_dict = None
# Array of indices for linking with mxy and contours
targetIndex = None


In [ ]:
# Variable initial value settings
filepath = ""
FPS = 20
T_R = -1
T_B = 0
T = 1600  # Number of frames to analyze
ADD_TIME_SEC = [22]  # Time when solution was exchanged
FIT_FRAME = "Auto"  # Frames for fitting; default is automatic
MAX_CELL_SIZE, MIN_CELL_SIZE = 300, 100

ini_params = [
    filepath,
    FPS,
    T_R,
    T_B,
    T,
    ADD_TIME_SEC,
    FIT_FRAME,
    MIN_CELL_SIZE,
    MAX_CELL_SIZE,
]

# Option for whether to perform aberration correction on images
flatten_image = True


In [ ]:
def cellfinder(img):
    # Input: numpy 16-bit integer array
    # Output: numpy boolean array (0, 1)

    # Gaussian filter
    kernel2 = np.array([[1, 2, 1],
                        [2, 4, 2], 
                        [1, 2, 1]]) / 16
    dst = cv2.filter2D(img, ddepth=-1, kernel=kernel2)  #convolve

    # Convert to 8-bit
    minvalue_u16 = np.amin(dst)
    maxvalue_u16 = np.amax(dst)
    dst = ((dst - minvalue_u16) / (maxvalue_u16 - minvalue_u16) * 255).astype(np.uint8)

    # Adaptive Otsu binarization
    # Set the computation radius
    radius = 10
    selem = disk(radius)  # Create a disk-shaped structuring element
    threshold_local_otsu = rank.otsu(
        image=dst, footprint=selem
    )  # Adaptive Otsu binarization
    # Disable thresholds that are too low, as they can be forcibly calculated from the background
    threshold_local_otsu[threshold_local_otsu <= np.mean(dst)] = 2**8 - 1
    binary_image = dst >= threshold_local_otsu

    return binary_image


In [ ]:
# njit function for fast computation of contour centroids
@njit(cache=True)
def gravitypoint(image_i, xmin, ymin, xmax, ymax, cellsize):
    # image_i is assumed to be a boolean image
    gx, gy = 0, 0
    for x in range(0, xmax - xmin + 1):
        for y in range(0, ymax - ymin + 1):
            if image_i[x, y]:
                gx += x + xmin
                gy += y + ymin
    return gx / cellsize, gy / cellsize


def cellSelector(
    t, image0, mxy
):  # Selects cells to analyze and creates an array where each cell is drawn with its cell number (background is -1); appends elements to mxy
    
    # Counting up cells that satisfy the conditions
    k = 0
    # Initialize image0 with -1 (since cell numbers start from 0)
    image0.fill(-1)
    mxy_t = []
    for i in range(hierarchy.shape[1]):
        ymin, xmin, height, width = cv2.boundingRect(contours[i])
        xmax, ymax = xmin + width - 1, ymin + height - 1

        # Skip if the contour is clearly smaller than MIN_CELL_SIZE
        if height * width < MIN_CELL_SIZE:
            continue

        cv2.drawContours(
            image0[: xmax + 1, : ymax + 1], contours, i, i, -1
        )  # Draw only the corresponding cell with value i on the zero matrix
        image_i = image0[xmin : xmax + 1, ymin : ymax + 1] == i
        cellsize = np.sum(image_i)
        if MIN_CELL_SIZE <= cellsize <= MAX_CELL_SIZE:
            # Extract only the necessary parts of the cell shape as a boolean matrix
            # Compute centroid
            gx, gy = gravitypoint(image_i, xmin, ymin, xmax, ymax, cellsize)
            # Exclude cases where no cell exists at the centroid position,
            # as this suggests the cell was not correctly recognized
            if image0[int(gx + 0.5), int(gy + 0.5)] != i:
                continue

            # Record cell area
            cellsize_dict[t][i] = cellsize

            # Record centroid, findContours index, and hierarchy index
            mxy_t.append([float("nan"), gx, gy, i])

            k += 1  # Count the number of cells to be observed
    mxy[t] = np.array(mxy_t, dtype=np.float64)
    return k


In [ ]:
def momentCalculator(t, mxy, targetIndex):
    # First, calculate the area of the obtained contours and select only those that qualify
    # Calculate moments from the obtained contours and record the centroid, approximate line, and normal in mxy
    # Number organization is fully delegated to numOrganizer!!
    for k in range(mxy[t].shape[0]):
        i = int(mxy[t][k][3])
        if i < 0:
            break

        ymin, xmin, height, width = cv2.boundingRect(contours[i])
        xmax, ymax = xmin + width - 1, ymin + height - 1

        image_i = image0[xmin : xmax + 1, ymin : ymax + 1] == i

        # Calculate moments and store them in mxy
        gx = mxy[t][k][1]  # x-coordinate of centroid
        gy = mxy[t][k][2]  # y-coordinate of centroid

        Ix = np.reshape(np.arange(xmin, xmax + 1) - gx, [xmax - xmin + 1, 1])
        Ixx = np.sum((Ix**2) * image_i)  # Moment of inertia around the y-axis
        Iy = np.reshape(np.arange(ymin, ymax + 1) - gy, [1, ymax - ymin + 1])
        Iyy = np.sum(image_i * (Iy**2))  # Moment of inertia around the x-axis
        Ixy = np.sum(image_i * Iy * Ix)

        moments = 2 * Ixy / (Ixx - Iyy)
        # The method for calculating the cell normal varies depending on the magnitude of Ixx/Iyy
        if Ixx / Iyy > 1 and moments != 0:
            mxy[t][k][0] = (-1 - np.sqrt(1 + moments**2)) / moments
        elif Ixx / Iyy < 1 and moments != 0:
            mxy[t][k][0] = (-1 + np.sqrt(1 + moments**2)) / moments
        else:  # When Ixx=Iyy, the target shape is symmetric about y=x, x-axis, and y-axis; when moment==0, the slope is 0
            mxy[t][k][0] = 0

        if np.isnan(
            mxy[t][k][0]
        ):  # If the normal slope becomes NaN during calculation (i.e., perpendicular to the x-axis), assign a sufficiently large number to avoid errors
            mxy[t][k][0] = 10**3

        if t == 0:
            targetIndex[0][k] = i  # Record targetIndex when t=0


In [ ]:
# Compares with the image from n frames ago to calculate cell movement from centroid positions
# Note: If this function is called multiple times, call it in ascending order of n
# This is because comparisons with more recent frames are more reliable!
def numOrganizer(t, mxy, targetIndex, n=1):
    """
        0. Create re_diffs using mxy[t-n][indices_re, 1:3] and mxy[t][noDupli, 1:3]
        1. Iterate through the array; if re_distance[i][j] < MAX_DRIFT_DISTANCE, then pair[i].append(j) and pair[j].append(i)
        2. If pair[i] has exactly one element and pair[*pair[i]] also has exactly one element, the condition is satisfied
        3. Assign the number from targetIndex[t-n][y] to targetIndex[t][x] that satisfies the above conditions
    """
    if n > t or n < 0:
        return

    # Maximum distance the centroid is expected to move in one frame
    MAX_DRIFT_DISTANCE = 7

    # Mask for extracting centroid coordinates from mxy for cells not yet stored in targetIndex[t]
    noDupli = np.isin(mxy[t][:, 3], targetIndex[t][targetIndex[t] != -1], invert=True)
    # Mask for cells in targetIndex that are not found at t but were found at t-n
    reFinder = (targetIndex[t] == -1) & (targetIndex[t - n] != -1)
    # Skip if no new cells can be found
    if sum(reFinder) == 0:
        return

    # Search for cells where reFinder is True in mxy
    # Look up the numbers in targetIndex[t-n][reFinder] from mxy[t-n][:, 3], preserving order
    # This relies on the fact that cell numbers in mxy are always in ascending order
    indices_re = np.searchsorted(mxy[t - n][:, 3], targetIndex[t - n][reFinder])

    # Calculate distances between centroids at t and t-n
    re_diffs = np.expand_dims(mxy[t - n][indices_re, 1:3], axis=1) - np.expand_dims(
        mxy[t][noDupli, 1:3], axis=0
    )
    # Rows correspond to the 1st term of re_diffs, columns to the 2nd term
    re_distance = np.sqrt(
        np.sum(re_diffs**2, axis=-1)
    )  # Matrix of inter-centroid distances computed for all combinations

    
    mxy_t_nums = mxy[t][noDupli, 3]
    fastNumOrganizer(
        t, targetIndex, re_distance, MAX_DRIFT_DISTANCE, reFinder, mxy_t_nums
    )


@njit(cache=True)
def fastNumOrganizer(
    t, targetIndex, re_distance, MAX_DRIFT_DISTANCE, reFinder, mxy_t_nums
):
    n_rows, n_cols = re_distance.shape
    # re_distance: rows correspond to t-n, columns correspond to t
    pairs = List()
    pairs.append(List([List.empty_list(types.int64) for _ in range(n_rows)]))
    pairs.append(List([List.empty_list(types.int64) for _ in range(n_cols)]))

    # Straightforwardly scan re_distance and record correspondences where the inter-centroid distance is less than MAX_DRIFT_DISTANCE
    for i in range(n_rows):
        for j in range(n_cols):
            if re_distance[i][j] < MAX_DRIFT_DISTANCE:
                pairs[0][i].append(j)
                pairs[1][j].append(i)
    # Inside numba, assignment via access like targetIndex[t][reFinder][i] (numeric, boolean array, numeric) is not possible
    idx = np.arange(targetIndex.shape[1])[reFinder]
    for i in range(n_rows):
        # Do not assign if the element does not have exactly one pair, or if its pair does not have exactly one pair
        if len(pairs[0][i]) != 1:
            continue

        j = pairs[0][i][0]

        if len(pairs[1][j]) != 1:
            continue

        targetIndex[t, idx[i]] = np.int32(mxy_t_nums[j])


In [ ]:
# Function for retrieving mxy information of a cell with contour number h at a given t
def mxylookup(h, t):
    if h == -1:
        return
    ng, ok = -1, mxy[t].shape[0]
    while abs(ok - ng) > 1:
        mid = (ok + ng) // 2
        if mxy[t][mid, 3] == -1 or mxy[t][ mid, 3] >= h:
            ok = mid
        else:
            ng = mid
    return ok

In [ ]:
# Function to calculate the normal line
@njit(cache=True)
def normal_line(t, mxy_t, xylim, i_mxy):
    xmin, xmax, ymin, ymax = xylim
    # Plot the normal line and record its information in lineList
    # Calculate y from x
    # Process all at once using array operations
    xrange = np.arange(xmin, xmax + 1)
    temp = mxy_t[i_mxy, 0] * (xrange - mxy_t[i_mxy, 1]) + mxy_t[i_mxy, 2]
    temp_fromX = (np.vstack((xrange, temp)) + 0.5).astype("int32")

    minmask = temp_fromX[1] >= ymin
    maxmask = temp_fromX[1] <= ymax
    line_fromX = temp_fromX[:, maxmask & minmask]

    # Calculate x from y
    yrange = np.arange(ymin, ymax + 1)
    if (
        mxy_t[i_mxy, 0] == 0
    ):  # To prevent bugs, assign a sufficiently small value when the slope is 0
        m = 0.0001
    else:
        m = mxy_t[i_mxy, 0]

    temp = mxy_t[i_mxy, 1] + (yrange - mxy_t[i_mxy, 2]) / m
    temp_fromY = (np.vstack((temp, yrange)) + 0.5).astype(
        "int32"
    )  # Combine with vstack and round to nearest integer
    maxmask = temp_fromY[0, :] <= xmax
    minmask = temp_fromY[0, :] >= xmin
    line_fromY = temp_fromY[:, maxmask & minmask]

    # Return the normal line (may contain duplicate elements)
    return np.hstack((line_fromX, line_fromY))


# Function that accelerates the bottleneck of fluoTotaler using numba
@njit(cache=True)
def fastFITotaler(
    t, fluoIntensity, changeFlag, mxy_t, image1, xylim, i, i_mxy, image_i, lineList_np
):
    xmin, xmax, ymin, ymax = xylim

    # Move the calculated normal line over the cell to acquire data
    Xs = lineList_np[0]
    Ys = lineList_np[1]

    if (
        abs(mxy_t[i_mxy, 0]) >= 1
    ):  # When the absolute value of the normal slope is >= 1, move the normal in the y-direction to fill the cell
        lmax_x = np.amax(
            Xs
        )  # Maximum x value of the normal line passing through the centroid
        lmin_x = np.amin(Xs)
        for j in range(xmin - lmax_x, xmax - lmin_x + 1):
            fIs = []
            # Retrieve brightness values along the shifted normal line
            for k in range(lineList_np.shape[1]):
                x, y = lineList_np[:, k]
                if xmin <= (x + j) <= xmax and image_i[x + j - xmin, y - ymin]:
                    fIs.append(image1[x + j, y])
            if not fIs:  # Cancel if all mask values are False
                continue

            # Average brightness value along the normal line
            fI = np.sum(np.array(fIs)) / len(fIs)

            # Adjust reading direction
            if (
                changeFlag[i, 0] == 1
            ):  # When changeFlag is 1, store average brightness in fluoIntensity in ascending order
                # Setting 0 at the centroid allows accurate alignment in later processing
                fluoIntensity[t][i][j] = fI
            else:  # When changeFlag is not 1, store average brightness in fluoIntensity in descending order
                # Setting 0 at the centroid allows accurate alignment in later processing
                fluoIntensity[t][i][-j] = fI

    else:  # When the absolute value of the normal slope is < 1, move the normal in the x-direction to fill the cell
        lmax_y = np.amax(
            Ys
        )  # Maximum y value of the normal line passing through the centroid
        lmin_y = np.amin(Ys)
        for j in range(ymin - lmax_y, ymax - lmin_y + 1):
            fIs = []
            # Retrieve brightness values along the shifted normal line
            for k in range(lineList_np.shape[1]):
                x, y = lineList_np[:, k]
                if ymin <= (y + j) <= ymax and image_i[x - xmin, y + j - ymin]:
                    fIs.append(image1[x, y + j])
            if not fIs:  # Cancel if all mask values are False
                continue

            # Average brightness value along the normal line
            fI = np.sum(np.array(fIs)) / len(fIs)

            # Adjust reading direction
            if (
                changeFlag[i, 0] == 1
            ):  # When changeFlag is 1, store average brightness in fluoIntensity in ascending order
                fluoIntensity[t][i][j] = fI
            else:  # When changeFlag is not 1, store average brightness in fluoIntensity in descending order
                fluoIntensity[t][i][-j] = fI


def fluoTotaler(
    t, fluoIntensity, targetIndex, changeFlag
):  # Function to draw normal lines and aggregate brightness values

    for i in range(targetIndex[t].shape[0]):  # Observe only cells stored in targetIndex

        h = int(targetIndex[t][i])  # Explicitly cast the i-th targetIndex to an integer

        if h == -1:
            continue  # Skip when h is -1, as no corresponding cell can be specified

        i_mxy = mxylookup(
            h, t
        )  # Since i in mxy and targetIndex have no relation, associate them via h
        ymin, xmin, height, width = cv2.boundingRect(contours[h])
        xmax, ymax = xmin + width - 1, ymin + height - 1

        image_i = image0[xmin : xmax + 1, ymin : ymax + 1] == h

        if t > 0:
            # Assume the rotation speed of a tethered cell is limited to 2π/3 [rad] per frame
            if ((changeFlag[i, 1] - 1) * (mxy[t][i_mxy, 0] - 1)) < 0 and abs(
                math.atan(changeFlag[i, 1]) - math.atan(mxy[t][i_mxy, 0])
            ) < math.pi * 2 / 3:
                # Flip changeFlag when crossing the y=x boundary where the reading direction reverses
                changeFlag[i, 0] *= -1

        if mxy[t][i_mxy, 0] != 1:
            # Update the value (do not update when on the y=x boundary; retain the previous value)
            changeFlag[i, 1] = mxy[t][i_mxy, 0]

        xylim = np.array([xmin, xmax, ymin, ymax])
        mxy_t = mxy[t]
        # Calculate the normal line and select only unique (x, y) combinations
        lineList_np = np.unique(normal_line(t, mxy_t, xylim, i_mxy), axis=1)
        # Use the normal line to assign values to fluoIntensity
        fastFITotaler(
            t,
            fluoIntensity,
            changeFlag,
            mxy_t,
            image1,
            xylim,
            i,
            i_mxy,
            image_i,
            lineList_np,
        )


In [ ]:
# Function to obtain the time-series changes in brightness values of the poles and cytoplasm from fluoIntensity
def pole_cytoTotaler(i, localizationTargetIndex, fluoIntensity):
    h = [r[0] for r in localizationTargetIndex][i]
    nonzero = fluoIntensity[:, h, :] != 0
    if localizationTargetIndex[i][1] < 0 and localizationTargetIndex[i][2] + 1 >= 0:
        pole = (
            np.sum(fluoIntensity[:, h, localizationTargetIndex[i][1] :], axis=1)
            + np.sum(fluoIntensity[:, h, : localizationTargetIndex[i][2] + 1], axis=1)
        ) / (
            np.sum(nonzero[:, localizationTargetIndex[i][1] :], axis=1)
            + np.sum(nonzero[:, : localizationTargetIndex[i][2] + 1], axis=1)
        )

    else:
        pole = np.sum(
            fluoIntensity[
                :, h, localizationTargetIndex[i][1] : localizationTargetIndex[i][2] + 1
            ],
            axis=1,
        ) / (
            np.sum(
                nonzero[
                    :, localizationTargetIndex[i][1] : localizationTargetIndex[i][2] + 1
                ],
                axis=1,
            )
        )

    cyto = fluoIntensity[:, h, localizationTargetIndex[i][3]]

    return pole, cyto


In [ ]:
# Define the function to be fitted
def func(x, A, a, a2, x1, x2, A1, mu, sigma):
    # Function combining two sigmoids and one Gaussian
    return A*(1/(1 + np.exp(-a*(x-x1)))+1/(1 + np.exp(a2*(x-x2)))-1)+A1*np.exp(-(x-mu)**2/sigma**2)

# Two sigmoid functions
def sigmoid(x, A, a, a2, x1, x2, A1, mu, sigma):
    # Unused parameters are also included so that popt can be passed through as-is.
    return A*(1/(1 + np.exp(-a*(x-x1)))+1/(1 + np.exp(a2*(x-x2)))-1)

# Gaussian function
def gaussian(x, A, a, a2, x1, x2, A1, mu, sigma):
    # Unused parameters are also included so that popt can be passed through as-is.
    return A1*np.exp(-(x-mu)**2/sigma**2)


In [ ]:
# Function to correct for photobleaching of the obtained brightness values
def with_fad_cyto(x, a, b):
    return np.exp(a * x + b)


# Determine the coefficients a and b of with_fad_cyto by fitting
def fad_correct(cyto):
    data = cyto[T_B:]
    nonzero = (
        data != 0
    )  # In data (assumed to be cyto), 0 indicates unanalyzable points and should be excluded as fitting them is meaningless
    x = np.arange(data.shape[0])[nonzero]
    popt, pcov = curve_fit(
        with_fad_cyto, x, data[nonzero], p0=[0, np.log(nonzero[0])], maxfev=100000
    )
    return popt


In [ ]:
def perfect_convolve(array, window_size):
    
    array_mean=np.zeros(array.shape[0])*float('nan')#Output array. Initialized with all NaN values.
    
    for i in range(array.shape[0]):
        window_uphalf=np.minimum(i+math.ceil(window_size/2),array.shape[0]-1)#Upper bound of the window. Uses minimum to prevent out-of-bounds access.
        window_dwhalf=np.maximum(i-(window_size-math.ceil(window_size/2)),0)#Lower bound of the window. Uses maximum to prevent going below 0.
        perfect_window=np.logical_not(np.isnan(array[window_dwhalf:window_uphalf]))#Include only non-NaN values in the calculation.
        valsum=np.sum(array[window_dwhalf:window_uphalf][perfect_window])
        windowsum=np.sum(perfect_window)
        array_mean[i]=valsum/windowsum

    return array_mean

In [ ]:
# PowerPoint slide creation functions

SLIDE_WIDTH, SLIDE_HEIGHT = 12193200, 6858000
# X, Y coordinates of the slide center (origin at top-left)
IMG_CENTER_X, IMG_CENTER_Y = SLIDE_WIDTH/2, SLIDE_HEIGHT/2
# Aspect ratio of the slide
SLIDE_ASPECT_RATIO = SLIDE_WIDTH / SLIDE_HEIGHT


# Adds a slide to the received presentation object and returns the added slide object.
def add_slide(prs):
    # Add a blank slide (ID=6 is a blank slide)
    blank_slide_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_slide_layout)
    return slide

# Paste an image at the center of the slide
def add_picture(slide, img_file):
    # Get image size to obtain aspect ratio
    im = Image.open(img_file)
    im_width, im_height = im.size
    aspect_ratio = im_width/im_height

    # Branch processing based on aspect ratio of slide and image
    # If the image is wider, expand it to full width
    if aspect_ratio > SLIDE_ASPECT_RATIO:
        img_display_width = SLIDE_WIDTH
        img_display_height = img_display_width / aspect_ratio
    else: # If the image is taller, expand it to full height
        img_display_height = SLIDE_HEIGHT
        img_display_width = img_display_height * aspect_ratio
    # Calculate the top-left coordinates of the image when centering
    left = IMG_CENTER_X - img_display_width / 2
    top = IMG_CENTER_Y - img_display_height / 2

    # Add the image to the slide
    if aspect_ratio > SLIDE_ASPECT_RATIO:
        slide.shapes.add_picture(img_file, left, top, width = img_display_width)
    else:
        slide.shapes.add_picture(img_file, left, top, height = img_display_height)

    return slide

def add_results(slide,graph_file,fit_file,row):
    # Paste 3 sets of localization graphs and fitting results per slide into PowerPoint
    # This function is not very general-purpose. Assumes localization graphs are 1500x300 and fitting results are 1200x400.
    img = Image.open(graph_file)
    img_width, _ = img.size
    
    top=SLIDE_HEIGHT/4*(row+1)
    
    slide.shapes.add_picture(graph_file,SLIDE_WIDTH/3+img_width*7/8, top, width =SLIDE_WIDTH*5/7 )
    slide.shapes.add_picture(fit_file,-1*SLIDE_WIDTH/30, top, height =SLIDE_HEIGHT/4 )
    
    return slide

In [ ]:
# Function for outputting analysis log text
def logtxt(params, out_dir, analyze_ratio: str):
    now = datetime.datetime.now()
    date = now.strftime("%Y%m%d")[2:]
    file = open(os.path.join(out_dir, f"analysis_log_{date}.txt"), "w")
    (
        filepath,
        FPS,
        T_R,
        T_B,
        T,
        ADD_TIME_SEC,
        FIT_FRAME,
        MIN_CELL_SIZE,
        MAX_CELL_SIZE,
    ) = params
    analyze_log = (
        f"analyzed_date:{now.strftime('%Y-%m-%d')}\n"
        # Getting the filename in Jupyter Notebook is cumbersome,
        # so we retrieve the name of the directory containing the file instead.
        f"analysis_file:{os.getcwd().split(os.sep)[-1]}\n"
        f"analysis_target:{filepath}\n"
        f"FPS:{FPS}\n"
        f"CheR:~up to frame {T_R}\n"
        f"CheB:frames {T_B}~{T}\n"
        f"solution_exchange:{','.join(map(str,ADD_TIME_SEC))} sec\n"
        f"fitting:{'auto' if FIT_FRAME=='Auto' else f'uniform {FIT_FRAME} frames'}\n"
        f"assumed_cell_area:{MIN_CELL_SIZE}pix~{MAX_CELL_SIZE}pix\n"
        f"final_selected_data/avg_analyzed_cells:{analyze_ratio}\n"
    )
    file.write(analyze_log)
    file.close()
    print("Analysis log file has been output.")
    return


In [ ]:
# Load Tiff for image correction such as aberration
# Assumes the image size for analysis is 1152*1152 pix
# If the image size is unexpected, an error will occur later, so a warning message is printed
# If a warning appears, either set flatten_image to False or capture a correction Tiff.

if flatten_image:
    """
    # Background
    Fill the chamber with 10000x bead solution (for focusing) + buffer and capture fluorescence
    Process into an average image of 100 frames using ImageJ's Image > Stacks > Z Project
    """
    background_filepath = r"AVG_background.tif"
    with TiffFile(background_filepath) as tif:
        bgmmap = tif.asarray(out="memmap")  # out="memmap" returns a memory map
        # Convert to 64-bit floating point
        bg = np.array(bgmmap).astype(np.float64)
    

    """
    # Laser aberration and other optical distortions in dark field
    Flow fluorescent dye + beads (for focusing) + buffer into the chamber and capture fluorescence
    Process into an average image of 100 frames using ImageJ's Image > Stacks > Z Project
    """
    illumination_pattern_filepath = r"AVG_Alexa_illumination_pattern.tif"
    with TiffFile(illumination_pattern_filepath) as tif:
        ipmmap = tif.asarray(out="memmap")  # out="memmap" returns a memory map
        # Convert to 64-bit floating point
        ip = np.array(ipmmap).astype(np.float64)


# main

## File loading and configuration of various settings

In [ ]:
# First, use GUI to configure various settings
ini_params= htagui.valueget(ini_params)  
filepath,FPS,T_R,T_B,T,ADD_TIME_SEC,FIT_FRAME,MIN_CELL_SIZE,MAX_CELL_SIZE=ini_params
print(filepath)
print(f"Configured values: FPS={FPS}, CheR up to frame {T_R}, CheB from frame {T_B}, analysis duration {T} frames, solution exchange time { ','.join(map(str,ADD_TIME_SEC)) } sec")
print("Frames for fitting:", FIT_FRAME, ", MINCELLSIZE=%d, MAXCELLSIZE=%d" %(MIN_CELL_SIZE,MAX_CELL_SIZE))
OUTPUT_FILE_DIR=os.path.dirname(filepath)# Set the output directory for various files to the same level as the analysis file
file_name=pathlib.Path(filepath).stem # Get the file name


In [ ]:
# Read Tiff
print("Reading Tiff file...")
tif=TiffFile(filepath)
mmap = tif.asarray(out="memmap")  # out="memmap" returns a memory map

if flatten_image:  # When applying aberration correction
    if (
        mmap.shape[1:] != bg.shape
    ):  # When the loaded image and correction image sizes differ
        flatten_image = False  # Forcibly disable correction
        print("The analysis image and correction image sizes do not match!")

print("completed")


## Estimation of Cellular Fluorescence Intensity

In [ ]:
# Process to extract necessary information from images
# Detect cells from images, track over time, and aggregate brightness values
start = time.time()
for t in range(T):  # Analyze frame by frame
    # Skip aggregation during filter switching
    if T_R <=t < T_B-1:
        print(f"{t=}:skip")
        continue
    
    # Apply correction if correction option is enabled
    if flatten_image:
        image1_ = (np.array(mmap[t])-bg)/((ip-bg)/np.amax(ip-bg))+bg # Retrieve image from memory map & apply correction
    else:
        image1_ = np.array(mmap[t])  # Retrieve image from memory map
    
    image1 = image1_.astype(np.uint16)
    image0 = np.ones(image1.shape)*(-1)
    image2=cellfinder(image1)
    image2_U8 = np.uint8(image2)  # Convert dtype of image2 to 8U for use with findContours
    contours, hierarchy = cv2.findContours(
        image2_U8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # contours holds coordinate information of contours, hierarchy returns hierarchical info (not very meaningful here since RETR_EXTERNAL is used)
    if t == 0:  # Analysis at t=0
        
        # List of dictionaries recording cell sizes; index corresponds to contour number
        cellsize_dict=[{} for _ in range(T)]
        # List of numpy arrays recording normal slope, centroid x/y coordinates, and contour index assigned by findContours, per cell and per frame
        mxy = [[] for _ in range(T)]
        
        cellNum = cellSelector(t,image0,mxy)  # Record cell count
        print(f"found {cellNum} cells")
        # Index for continuously tracking cells detected at t=0 throughout the observation period; -1 means no cell is specified
        targetIndex = np.ones((T, cellNum))*(-1)
        
        maxCelllength = MAX_CELL_SIZE//3  # maxCelllength must be no greater than MAX_CELL_SIZE
        fluoIntensity = np.zeros((T, cellNum, maxCelllength))
        changeFlag = np.ones((cellNum, 2))
        momentCalculator(t, mxy, targetIndex)  # Calculate various moments
        fluoTotaler(t, fluoIntensity, targetIndex, changeFlag)  # Aggregate brightness values
        print(f"{t=}")

    if t != 0:  # Analysis at t!=0
        # When the frame referenced by numOrganizer is during filter switching
        
        cellSelector(t,image0,mxy)
        momentCalculator(t, mxy, targetIndex)  # Calculate various moments
        # Do not reference frames during filter switching
        # Assuming there is not much drift, reference the frame just before switching
        n_main= T_B-T_R+1 if t==T_B else 1
        n_sub= T_B-T_R+25 if T_B <= t < T_B+25 else 25

        numOrganizer(t, mxy, targetIndex,n=n_main)  # Associate detected contours with the image 1 frame prior
        numOrganizer(t, mxy, targetIndex,n=n_sub)  # Associate detected contours with the image 25 frames prior
        fluoTotaler(t, fluoIntensity, targetIndex, changeFlag)  # Aggregate brightness values
        print(f"{t=}")
end = time.time()
print("Time taken up to brightness aggregation:", end-start, "seconds")

## Fitting

In [ ]:
# Search for localized cells using fitting

temp_mydir = tempfile.mkdtemp()# Create a temporary directory

fit_result_dir = os.path.join(temp_mydir,"fit_result/")
# Create a fit_result directory inside the temporary directory
os.makedirs(fit_result_dir, exist_ok=True)# Allow creation even if a directory with the same name already exists

graph_result_dir = os.path.join(temp_mydir,"graph_result/")
# Create a graph_result directory inside the temporary directory
os.makedirs(graph_result_dir, exist_ok=True)# Allow creation even if a directory with the same name already exists

localizationTargetIndex = []

# Get the number of digits so that files are sorted correctly by name when saved
if targetIndex.shape[1]==1:
    digits=1 # Prevent bugs from log10(0)
else:
    digits=int(np.log10(targetIndex.shape[1])+1)# Store the maximum number of digits

for i in range(fluoIntensity.shape[1]):  # Observe only cells stored in targetIndex
    # Branch depending on whether fitting frame is set to auto or manual
    if FIT_FRAME=="Auto":
        # Automatically find the fitting frame
        Z=np.amax(fluoIntensity[int(ADD_TIME_SEC[0]*FPS):,i,:],axis=1).copy()
        Z[Z==0]=float('nan')
        _pc=perfect_convolve(Z,20)
        # Replace nan with 0 to avoid interfering with argmax
        _pc[np.isnan(_pc)]=0
        fit_frame=np.argmax(_pc)+ int(ADD_TIME_SEC[0]*FPS)
    else:
        fit_frame=FIT_FRAME
    
    yscalemax=np.amax(fluoIntensity[:,i]*1.1)
    
    if flatten_image:
        image1_ = (np.array(mmap[fit_frame])-bg)/((ip-bg)/np.amax(ip-bg))+bg # Retrieve image from memory map & apply correction
    else:
        image1_ = np.array(mmap[fit_frame])  # Retrieve image from memory map
    
    image1 = image1_.astype(np.uint16)
    # When a high-throughput microscope Tiff is processed in ImageJ, the data type becomes >u2, causing overflow during convolution. Explicitly convert to uint16.
    image2=cellfinder(image1)
    image2_U8 = np.uint8(image2)  # Convert dtype of image2 to 8U for use with findContours
    contours, hierarchy = cv2.findContours(image2_U8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    image0 = np.zeros(image1.shape)
    h = int(targetIndex[fit_frame][i])
    if h==-1:continue
    cv2.drawContours(image0, contours, h, 1, -1)  # Draw only the target cell as 1 on a zero matrix
    avgMask = fluoIntensity[fit_frame:fit_frame+10, i, 0] != 0

    if np.sum(avgMask) < 10:  # Skip if there are insufficient brightness values for averaging
        #print(i,fit_frame,h,"too little sample")
        continue
    
    imgav = np.sum(fluoIntensity[fit_frame:fit_frame+10, i, :],axis=0)/np.sum(avgMask)  # Cumulative average of the first 10 frames

    shift = np.sum(imgav[maxCelllength//2:] != 0)
    # shift := np.sum(imgav[maxCelllength//2:]!=0) is the number of brightness values on the right side.
    # Shifting by this amount allows displaying a graph starting from position 0.
    # fluoIntensity has the centroid at position 0, which is hard to read. Shift the start position for clarity.
    imgf = np.roll(imgav, (shift+1))
    
    # Initial parameters (set per-data initial values close to expected values to reduce computation and avoid poor fitting)
    # Parameter order: coefficients A, a, a2, x1, x2, A1, mu, sigma of func
    parameter_initial = np.array(
        [imgav[0],  # Localization at the centroid is unlikely, so the brightness at centroid should be the cytoplasmic brightness
        1,  # Empirically, 1 seems most common
        1,  # Empirically, 1 seems most common
        # The position in the first half of imgf closest to A/2 should be x1
        np.argmin(((imgav[0]/2-imgf)[1:shift+1])**2),
        # The position in the second half of imgf closest to A/2 should be x2
        np.argmin(((imgav[0]/2-imgf)[shift+1:np.sum(imgav != 0)+1])**2)+shift+1,
        np.amax(imgf)-imgav[0],#
        np.argmax(imgf),# The peak of the Gaussian component should be where brightness is highest
        10])

    plt.figure(figsize=(12, 4))
    ax = plt.subplot(121)
    plt.plot(imgf)
    plt.axis([-1, np.sum(imgf != 0)+3, 0, yscalemax])# Set lower and upper bounds for the fitting graph axes
    
    
    # Optimization calculation
    try:
        # maxfev is the maximum number of iterations: default was 1600, but increased to 100,000 to handle non-convergence cases.
        # Added exception handling since even 100,000 iterations sometimes fail
        popt, pcov = curve_fit(func, np.arange(maxCelllength), imgf, p0=parameter_initial, maxfev=100000)
        
        # Rarely, popt may contain nan, which interrupts computation.
        # Raise ValueError to pass to exception handling and avoid interrupting fitting
        if any(np.isnan(popt)):
            raise ValueError("NaN values are included!")
    except:
        plt.title(f"cell{str(i).zfill(digits)}fitting failed")
        continue
    y = func(np.arange(maxCelllength), *popt)
    s = sigmoid(np.arange(maxCelllength), *popt)
    g = gaussian(np.arange(maxCelllength), *popt)

    plt.plot(np.arange(maxCelllength), y, 'r-',)
    plt.plot(np.arange(maxCelllength), s, 'g-',)
    plt.plot(np.arange(maxCelllength), g, 'm-',)

    # popt order: A, a, a2, x1, x2, A1, mu, sigma
    localmin = np.maximum(int(popt[6]-1*abs(popt[7])+0.5), 1)
    localmax = np.minimum(
        int(popt[6]+1*abs(popt[7])+0.5), np.sum(imgf != 0))
    # The cytoplasm (presumed) is where the difference from A is smallest
    cytodiff = (popt[0]-imgf)[int(shift*0.5):int(shift*1.5)+1]
    
    # Set localization conditions
    # Conditions in order: A1 > 1, before checking if localization region has brightness >= 1.2x cytoplasm,
    # popt[6] (Gaussian peak) can sometimes be an extreme value, so verify it is a valid index for y.
    # Also include condition that localization region brightness is >= 1.2x cytoplasmic brightness.
    
    # popt order: A, a, a2, x1, x2, A1, mu, sigma
    if (popt[5] >= 1 and 0 <= popt[6] < maxCelllength  and
            y[int(popt[6])] > s[int(popt[6])]*1.2 and 
            # Ensure the presumed localization range does not include the cytoplasm
            not (localmin < np.argmin(cytodiff**2)+int(shift*0.5) < localmax)
            ):
        
        print("Localization range is", localmin, "to", localmax)
        
        # localizationTargetIndex contains: cell number in TargetIndex, localization start, localization end, cytoplasm position
        # Revert the shift to apply to the unshifted fluoIntensity
        localizationTargetIndex.append([i, int(localmin)-(shift+1),
                                        int(localmax)-(shift+1),
                                        np.argmin(cytodiff**2)+int(shift*0.5)-(shift+1)])

        plt.vlines(np.argmin(cytodiff**2) + int(shift*0.5), yscalemax, 0, color="black")
        ax.axvspan(localmin, localmax, color="coral")
        plt.title(f"cell{str(i).zfill(digits)}")
        plt.subplot(122)
        xmin, xmax = np.amin(contours[h][:, 0, 1]), np.amax(
            contours[h][:, 0, 1])
        ymin, ymax = np.amin(contours[h][:, 0, 0]), np.amax(
            contours[h][:, 0, 0])
        plt.imshow((image0*image1)[xmin:xmax+1, ymin:ymax+1], cmap='gray')
        plt.title(str(fit_frame)+"[f]")
        plt.savefig(os.path.join(fit_result_dir,f"cell{str(i).zfill(digits)}.png")) # Save to file
        
        lti = len(localizationTargetIndex)-1
        pole, cyto = pole_cytoTotaler(lti, localizationTargetIndex, fluoIntensity)
        
        # Display localization graph
        plt.rcParams['figure.subplot.bottom'] = 0.15 # Prevent graph from being clipped
        plt.figure(figsize=(15, 3))
        plt.title(f"cell{str(i).zfill(digits)}")# Show cell number and fitted frame
        plt.minorticks_on()  # Minor ticks
        plt.grid(color='black', linestyle=':', linewidth=0.3, which='both', axis="x")
        plt.grid(color='black', linestyle=':', linewidth=0.3, which='major', axis="y")
        time_sec = np.arange(pole.shape[0])/FPS
        plt.axis([0, pole.shape[0]/FPS, 0, yscalemax])# Set lower and upper bounds for the localization graph axes
        plt.plot(time_sec, cyto, 'r', linewidth=0.7, label="cyto")
        plt.plot(time_sec, pole, linewidth=0.7, label="pole")
        plt.axvspan(0, ADD_TIME_SEC[0], color="palegreen")
        plt.xlabel("Time[s]")
        plt.ylabel("F.I[A.U.]")
        plt.legend()
        plt.savefig(os.path.join(graph_result_dir,f"cell{str(i).zfill(digits)}.png")) # Save to file
        plt.show()
        

    else:# If no localization is detected
        plt.title(f"cell{str(i).zfill(digits)}(no localization)")

        plt.subplot(122)
        xmin, xmax = np.amin(contours[h][:, 0, 1]), np.amax(
            contours[h][:, 0, 1])
        ymin, ymax = np.amin(contours[h][:, 0, 0]), np.amax(
            contours[h][:, 0, 0])
        plt.imshow((image0*image1)[xmin:xmax+1, ymin:ymax+1], cmap='gray')
        plt.title(f"{fit_frame}[f]")

# Preparation for result review
img_files = os.listdir(graph_result_dir)
imgs=sorted(img_files)# Sort filenames in alphabetical order
true_list=[True for _ in range(len(imgs))]# Create a bool array of all True values

# (Preparation to prevent bugs in txt writing even if localization duration calculation is skipped)
ld_true_list=[True for _ in range(len(imgs))]# Create a bool array of all True values

## Save cell distribution map

In [ ]:
# Display which number is assigned to which cell at the specified frame t
t = 0
fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(111)

if flatten_image:
    image1_ = (np.array(mmap[t]) - bg) / (
        (ip - bg) / np.amax(ip - bg)
    ) + bg  # Retrieve image from memory map & apply correction
else:
    image1_ = np.array(mmap[t])  # Retrieve image from memory map

image1 = image1_.astype(
    np.uint16
)  # The Tiff created by HCImage on CheB unit 3 has data type >u2, which causes overflow during convolution. Explicitly convert to uint16.
image2 = cellfinder(image1)
image2_U8 = np.uint8(image2)  # Convert dtype of image2 to 8U for use with findContours
contours, hierarchy = cv2.findContours(
    image2_U8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

rois = []  # For ROI Manager
image0 = np.zeros(image1.shape)
for i in range(targetIndex[t].shape[0]):

    h = int(targetIndex[t][i])
    if h == -1:
        continue  # Skip if h is -1, as no corresponding cell can be specified
    cv2.drawContours(
        image0, contours, h, 1, -1
    )  # Draw only the target cell as 1 on a zero matrix

    i_mxy = mxylookup(
        h, t
    )  # mxy and i in targetIndex have no direct relationship, so associate via h
    # Display cell name on the graph. Since matplotlib's coordinate origin is bottom-left, use y, x order for mxy centroid
    ax.text(mxy[t][i_mxy, 2], mxy[t][i_mxy, 1], str(i), color="lime")
    ymin, xmin, height, width = cv2.boundingRect(contours[h])
    xmax, ymax = xmin + width - 1, ymin + height - 1

    # Create ROI for ImageJ
    roi = roifile.ImagejRoi(
        roitype=roifile.ROI_TYPE.RECT,
        name=f"cell_{str(i).zfill(digits)}",
        version=228,
        top=xmin,
        left=ymin,
        bottom=xmax,
        right=ymax,
        position=t + 1,
    )
    rois.append(roi)
    # Draw a yellow bounding box around the cell
    r = patches.Rectangle(
        (ymin, xmin), height - 1, width - 1, fill=False, edgecolor="yellow", linewidth=1
    )
    ax.add_patch(r)
ax.imshow(image1, cmap="gray")
ax.set_title(f"t={t}[f] cellsize={MIN_CELL_SIZE}~{MAX_CELL_SIZE}[pix]")
plt.savefig(os.path.join(OUTPUT_FILE_DIR, "cell_distribution_map.png"))
roifile.roiwrite(
    os.path.join(OUTPUT_FILE_DIR, f"{file_name}_cell_map.zip"), rois, mode="w"
)


## Manual Visal Inspection

In [ ]:
# Launch result review GUI
true_list = htagui.true_local_select(imgs, true_list, temp_mydir)
selected_imgs = []
for i in range(len(imgs)):
    if true_list[i]:
        selected_imgs.append(imgs[i])
print(len(selected_imgs), "/", len(imgs), "is selected")


## Estimate localization duration

In [ ]:
# Calculate localization duration with photobleaching correction

ld_result_dir = temp_mydir + "/ld_result/"

if os.path.isdir(ld_result_dir):
    shutil.rmtree(
        ld_result_dir
    )  # If ld_result_dir already exists in temp_mydir, delete it along with its contents

# Create ld_result directory inside the temporary directory
os.makedirs(
    ld_result_dir, exist_ok=True
)  # Allow creation even if a directory with the same name already exists — though it shouldn't exist


duration_time_list = []
FImax = []
FIargmax = []

for i in range(len(localizationTargetIndex)):
    if true_list[
        i
    ]:  # Calculate & display localization duration only when true_list is True

        pole0, cyto0 = pole_cytoTotaler(i, localizationTargetIndex, fluoIntensity)
        # Explicitly copy before correction to prevent cyto, pole, and the underlying fluoIntensity from being overwritten
        cyto = cyto0.copy()
        pole = pole0.copy()

        popt = fad_correct(cyto)
        x = np.arange(cyto.shape[0])

        # Keep a separate copy of the uncorrected cyto for comparison
        cyto_nofixed = cyto.copy()

        pole = pole / np.exp(popt[0] * x)
        cyto[cyto != 0] = cyto[cyto != 0] / np.exp(
            popt[0] * x[cyto != 0]
        )  # Skip correction where cyto is 0, as those values are meaningless

        conv_pole = perfect_convolve(pole, 20)  # Apply moving average

        h = [r[0] for r in localizationTargetIndex][i]
        yscalemax = np.amax(fluoIntensity[:, h]) * 1.1
        time_sec = np.arange(pole.shape[0]) / FPS

        plt.figure(figsize=(15, 3))
        plt.title(f"cell{str(h).zfill(digits)}")
        plt.minorticks_on()
        plt.grid(color="black", linestyle=":", linewidth=0.3, which="both", axis="x")
        plt.grid(color="black", linestyle=":", linewidth=0.3, which="major", axis="y")
        plt.axis([0, pole.shape[0] / FPS, 0, yscalemax])
        plt.plot(time_sec, cyto_nofixed, linewidth=0.1, label="cyto(nofixed)")
        plt.plot(time_sec, cyto, "r", linewidth=0.5, label="cyto(fixed)")
        plt.plot(time_sec, conv_pole, "g", linewidth=1.2, label="pole(fixed&convolved)")

        duration_times = []
        fimax, fiargmax = [], []
        adts = [T_B / FPS] + ADD_TIME_SEC + [T / FPS]
        for j in range(1, len(ADD_TIME_SEC) + 1):
            conv_pole_2 = (
                conv_pole.copy()
            )  # Using =conv_pole would modify conv_pole itself, so use copy() to avoid that
            conv_pole_2[np.isnan(conv_pole)] = (
                0  # Fill NaN positions with a small value to prevent NaN from interfering with argmax
            )
            conv_pole_max = np.argmax(
                conv_pole_2[int(adts[j] * FPS) : int(adts[j + 1] * FPS)]
            ) + int(
                adts[j] * FPS
            )  # Frame with the highest pole brightness after solution exchange

            conv_pole_2[np.isnan(conv_pole)] = (
                2**16
            )  # Fill NaN positions with a large value (greater than 16-bit max) to prevent NaN from interfering with argmin

            # hdmask: mask for conv_pole_2 after the peak, where the rise is <= half of the maximum rise from before solution exchange
            hdmask = (
                conv_pole_2[conv_pole_max:]
                <= (
                    conv_pole[conv_pole_max]
                    + np.average(
                        conv_pole[int(adts[j - 1] * FPS) : int(adts[j] * FPS)][
                            ~np.isnan(
                                conv_pole[int(adts[j - 1] * FPS) : int(adts[j] * FPS)]
                            )
                        ]
                    )
                )
                / 2
            )
            if np.sum(hdmask) != 0:
                half_duration = np.amin(np.arange(conv_pole_max, pole.shape[0])[hdmask])
            else:
                half_duration = adts[j] * FPS
                print("WARNING: Could not calculate localization duration correctly.")

            duration_time = (
                half_duration / FPS - adts[j]
            )  # Output duration_time in seconds
            print("Localization duration=", duration_time)

            duration_times.append(str(f"{duration_time:.4f}"))

            fimax.append(conv_pole[conv_pole_max])
            fiargmax.append(conv_pole_max)
            # Fill localization duration range with sky blue
            plt.axvspan(adts[j], half_duration / FPS, color="skyblue")

        duration_time_list.append([str(h)] + duration_times)
        FImax.append(fimax)
        FIargmax.append(fiargmax)
        plt.xlabel("Time[s]")
        plt.ylabel("F.I[A.U.]")
        plt.legend()
        plt.savefig(
            os.path.join(ld_result_dir, f"cell{str(h).zfill(digits)}.png")
        )  # Save to file
        plt.show()

# Preparation for result review
ldimg_files = os.listdir(ld_result_dir)
imgs_ld = sorted(ldimg_files)  # Sort filenames in alphabetical order
ld_true_list = [True for _ in range(len(imgs_ld))]
selected_imgs0 = selected_imgs.copy()
duration_time_list0 = duration_time_list.copy()


## Manual Visal Inspection of localization Duration

In [ ]:
# Launch result review GUI
ld_true_list = htagui.ld_select(imgs_ld, ld_true_list, temp_mydir)
# selected_imgs was created during the previous manual check, so select from it
selected_imgs = []
duration_time_list = []
pole_max_list = []
pole_argmax_list = []
for i in range(len(selected_imgs0)):
    if ld_true_list[i]:
        selected_imgs.append(selected_imgs0[i])
        duration_time_list.append(duration_time_list0[i])
        pole_max_list.append(FImax[i])
        pole_argmax_list.append(FIargmax[i])
print(f"{len(selected_imgs)}/{len(selected_imgs0)}/{len(imgs)} is selected")


# Additional analysys

## Save Powerpoint

In [ ]:
# Create a PowerPoint file summarizing the analysis results

# Output file name
pptx_file_dir = os.path.join(
    OUTPUT_FILE_DIR, f"{file_name}_results_summary(auto-generated).pptx"
)
# file_name should be the Tiff file name

# Directory containing images
IMG_DIR = temp_mydir
prs = Presentation()
# Specify slide size
prs.slide_width = SLIDE_WIDTH
prs.slide_height = SLIDE_HEIGHT
img_files = selected_imgs

for i in range(len(selected_imgs)):
    name = selected_imgs[i]
    path_graph = os.path.join(IMG_DIR, "graph_result", name)
    path_fit = os.path.join(IMG_DIR, "fit_result", name)
    # Paste results of 3 cells per slide
    if i % 3 == 0:
        slide = add_slide(prs)

    add_results(slide, path_graph, path_fit, i % 3)


# Output the pptx file
prs.save(pptx_file_dir)
print("PowerPoint file has been created.")


## Save pole&cyto in text file

In [ ]:
txt_file_dir = os.path.join(
    OUTPUT_FILE_DIR, f"{file_name}_localization_fluorescence_intensity"
)
# file_name should be the Tiff file name

# Create a txt_file_dir directory inside the prepared temporary directory
os.makedirs(
    txt_file_dir, exist_ok=True
)  # Allow creation even if a directory with the same name already exists

k = 0

for i in range(len(localizationTargetIndex)):
    if true_list[i]:
        if ld_true_list[k]:
            pole, cyto = pole_cytoTotaler(i, localizationTargetIndex, fluoIntensity)

            data_write = np.empty((4, pole.shape[0]))

            data_write[0] = np.arange(pole.shape[0])
            data_write[1] = pole
            data_write[2] = cyto
            data_write[3] = pole - cyto

            data_write = data_write.T

            # To clarify: the 0th element of localizationTargetIndex[i] contains the cell number
            h = [r[0] for r in localizationTargetIndex][i]
            # Open file for writing data; create if it doesn't exist, overwrite if it does
            file = open(
                os.path.join(
                    txt_file_dir,
                    f"cell{str(h).zfill(digits)}_fluorescence_intensity.txt",
                ),
                "w",
            )
            # Output brightness distribution
            for ii in range(data_write.shape[0]):
                y = data_write[ii, :]
                for j in range(data_write.shape[1]):
                    file.write(
                        str(f"{y[j]:.4f}") + "\t"
                    )  # f'{y[j]:.4f}' means up to 4 decimal places
                file.write("\n")
            # Close the text file for brightness output
            file.close()

        k += 1

print("txt file export complete.")

# Also output the log file
analyze_ratio = f"{sum(ld_true_list)}/{np.average(np.sum(targetIndex!=-1,axis=1)[int(ADD_TIME_SEC[0]):T])}"
logtxt(ini_params, OUTPUT_FILE_DIR, analyze_ratio)


## Centroid trajectory

In [ ]:
# Centroid trajectory for all cells (detailed version)
# Displays the trajectory from t_start to t_end
t_start=0
t_end=T
check_with_ld=True

gxy=[]
for i in range(targetIndex[t].shape[0]):
    
    # When check_with_ld is True, skip cells that have no localization duration data
    if check_with_ld:
        if i not in [int(r[0]) for r in duration_time_list]:
            continue
    graviMovex=[]
    graviMovey=[]
    for f in range(t_start,t_end):
        t=f
        h=int(targetIndex[t][i])
        if h==-1:
            continue# Skip if h is -1, as no corresponding cell can be specified

        i_mxy = mxylookup(h,t) # mxy and i in targetIndex have no direct relationship, so associate via h
        graviMovex.append(mxy[t][i_mxy,1])
        graviMovey.append(mxy[t][i_mxy,2])
    
    if flatten_image:# If ip is defined, use illumination_pattern
        illumination_pattern=ip[int(np.average(graviMovex)),int(np.average(graviMovey))]/np.amax(ip)
    else:# If ip is not defined, insert NaN
        illumination_pattern=float('nan')
    
    gxy.append([np.average(graviMovex),
                np.average(graviMovey),
                illumination_pattern
                ])
    
    plt.figure(figsize=(5,5))
    ax=plt.gca()
    ax.set_aspect("equal",adjustable="datalim")
    plt.plot(graviMovey[:T_R],graviMovex[:T_R],"o",color="green",
            markersize=5,linewidth = 1)
    plt.plot(graviMovey[T_R:],graviMovex[T_R:],"o",color="tomato",
            markersize=5,linewidth = 1)
    plt.plot(graviMovey,graviMovex,"--",color="darkgray",
            markersize=5,linewidth = 1)
    plt.plot(np.average(graviMovey),np.average(graviMovex),"x",markersize=10,color="k")

    PtsOrPt="point"
    if np.sum(targetIndex[t_start:t_end,i]!=-1)!=1:
        PtsOrPt=PtsOrPt+"s"
    plt.title(f"cell{i} :{np.sum(targetIndex[t_start:t_end,i]!=-1)} {PtsOrPt} in {t_end-t_start} frames")
    plt.show()

In [ ]:
# Centroid trajectory for all cells (wide, simultaneous view)
# Displays the trajectory from t_start to t_end
t_start=0
t_end=T

plt.figure(figsize=(20,20)) # Prepare a large canvas; all plots are drawn here

# Track and plot the centroid position of all cells in targetIndex
for i in range(targetIndex[t].shape[0]):
    
    graviMovex=[]
    graviMovey=[]
    for f in range(t_start,t_end):
        t=f//15
        h=int(targetIndex[t][i])
        if h==-1:
            continue# Skip if h is -1, as no corresponding cell can be specified

        i_mxy = mxylookup(h,t)  # mxy and i in targetIndex have no direct relationship, so associate via h
        graviMovex.append(mxy[t][i_mxy,1])
        graviMovey.append(mxy[t][i_mxy,2])
    
    plt.plot(graviMovey,graviMovex,"o",color="gray",markersize=5,linewidth = 1)# Plot data points
    plt.plot(graviMovey,graviMovex,"--",color="darkgray",markersize=5,linewidth = 1)# Line connecting the points
    plt.plot(np.average(graviMovey),np.average(graviMovex),"x",markersize=10,color="green")# Plot average centroid position with x marker

plt.title("all cell centroid trajectory")
plt.xlim(0,image1.shape[0])
plt.ylim(image1.shape[1],0)
plt.show()

In [ ]:
# Centroid trajectory for a specified cell i (detailed version)
# Displays the trajectory from t_start to t_end
t_start=0
t_end=T
i=0

graviMovex=[]#np.zeros(t_end-t_start,2)
graviMovey=[]
for f in range(t_start,t_end):
    t=f#//15
    h=int(targetIndex[t][i])
    if h==-1:
        continue# Skip if h is -1, as no corresponding cell can be specified

    i_mxy = mxylookup(h,t)  # mxy and i in targetIndex have no direct relationship, so associate via h
    graviMovex.append(mxy[t][i_mxy,1])
    graviMovey.append(mxy[t][i_mxy,2])
plt.figure(figsize=(5,5))
ax=plt.gca()
ax.set_aspect("equal",adjustable="datalim") # Match aspect ratio to make drift intuitively easy to understand
plt.plot(graviMovey,graviMovex,"o",color="gray",markersize=5,linewidth = 1)# Plot data points
plt.plot(graviMovey,graviMovex,"--",color="darkgray",markersize=5,linewidth = 1)# Line connecting the points
plt.plot(np.average(graviMovey),np.average(graviMovex),"x",markersize=10,color="green")# Plot average centroid position with x marker

PtsOrPt="point"
if np.sum(targetIndex[t_start:t_end,i]!=-1)!=1:
    PtsOrPt=PtsOrPt+"s"
plt.title( f"cell{i} :{np.sum(targetIndex[t_start:t_end,i]!=-1)} {PtsOrPt} in {t_end-t_start} frames")
plt.show()

## Aggregation of parameters [cell area, average brightness etc]

In [ ]:
# Record cell size [1600f] and average brightness of CheR/CheB [check_time]

# Collect other parameters only for cells with calculated localization duration
check_with_ld=True
check_time=100

# Cell size check
if check_with_ld:
    # Create a zero matrix to record cell sizes
    cellsize_array_ld=np.zeros([len(duration_time_list),T])
    # Create a zero matrix to record total fluorescence intensity and cell length changes
    fI_length=np.zeros([len(duration_time_list),T,2])

# Analyze CheB-GFP fluorescence intensity from mCherry-CheR fluorescence intensity
for t in range(T):  
    
    j=-1 # Index starts from 0, but initialized to -1 because the program increments before storing cell area
    print(f"Calculating...{t+1}/{T}",end="\r")
    for i in range(targetIndex[t].shape[0]):

        h=int(targetIndex[t,i])
        
        if check_with_ld:
            if i not in [int(r[0]) for r in duration_time_list]:
                continue
            else:
                j+=1
        
        if h==-1:
            continue# Skip if h is -1, as it is a meaningless value
        
        if check_with_ld:
            cellsize_array_ld[j,t]=cellsize_dict[t][h]
            fI_length[j,t]=np.sum(fluoIntensity[t,i])/np.sum(fluoIntensity[t,i]!=0) , np.sum(fluoIntensity[t,i]!=0)# Average fluorescence intensity and length of the cell
        
        
print("\nCalculation complete.")

In [ ]:
# Combine in dataframe format

# CheB-related parameters
df_CheB=[
pd.DataFrame([[r[0]]+list(map(float,r[1:])) for r in duration_time_list],
             columns=["cell No","CheB duration time[s]"]+[f"CheB duration time_{i+1}[s]" for i in range(1,len(ADD_TIME_SEC))]),
pd.DataFrame(gxy,columns=["gx","gy","illumination pattern"]),
pd.DataFrame(np.sum(cellsize_array_ld[:,T_B:],axis=1)/np.sum(cellsize_array_ld[:,T_B:]!=0,axis=1),columns=["CheB cellsize[pix]"]),
pd.DataFrame(np.sum(fI_length[:,T_B:T_B+check_time,0],axis=1)/np.sum(cellsize_array_ld[:,T_B:T_B+check_time]!=0,axis=1),columns=[f"CheB F.I({check_time}f)"]),
pd.DataFrame(np.sum(fI_length[:,T_B:T_B+check_time,1],axis=1)/np.sum(cellsize_array_ld[:,T_B:T_B+check_time]!=0,axis=1),columns=["CheB cell length[pix]"]),
pd.DataFrame(pole_max_list,columns=["CheB pole max"]+[f"CheB pole max_{i+1}" for i in range(1,len(ADD_TIME_SEC))]),
pd.DataFrame(pole_argmax_list,columns=["CheB pole argmax[f]"]+[f"CheB pole argmax_{i+1}[f]" for i in range(1,len(ADD_TIME_SEC))])
]
# CheR-related parameters
df_CheR=[pd.DataFrame(np.sum(cellsize_array_ld[:,:T_R],axis=1)/np.sum(cellsize_array_ld[:,:T_R]!=0,axis=1),columns=["CheR cellsize[pix]"]),
pd.DataFrame(np.sum(fI_length[:,:check_time,0],axis=1)/np.sum(cellsize_array_ld[:,:check_time]!=0,axis=1),columns=[f"CheR F.I({check_time}f)"])
]

df_params= (df_CheB) if T_R<0 else (df_CheB + df_CheR)

# Concatenate all dataframes
dfx=pd.concat(df_params,axis=1)
# Display summary
print(dfx.head())
dfx.to_csv(os.path.join(OUTPUT_FILE_DIR,f"{file_name}_aggregated_data.csv"))
print("The above data has been saved.")

In [ ]:
# Pair plot
plt.figure(figsize=[10,10])
# Plot only necessary variables; cell number, centroid position, and illumination pattern are excluded
sns.pairplot(dfx,vars=[v for v in dfx.columns if v not in {"cell No","gx","gy","illumination pattern"}])
plt.show()

## Visualyze cell area

In [ ]:
# Display which number is assigned to which cell at the specified frame t
#---------------------------------------------
t=0
mincellsize=1
maxcellsize=2000
#---------------------------------------------

if flatten_image:
    image1_ = (np.array(mmap[t])-bg)/((ip-bg)/np.amax(ip-bg))+bg # Retrieve image from memory map & apply correction
else:
    image1_ = np.array(mmap[t])  # Retrieve image from memory map

image1=image1_.astype(np.uint16)# Explicitly convert data type to uint16
image2=cellfinder(image1)
image2_U8=np.uint8(image2)# Convert dtype of image2 to 8U for use with findContours
contours, hierarchy = cv2.findContours(image2_U8,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

image_cs=np.zeros(image1.shape)
cellsizelist=[]
# Count up cells that satisfy the conditions
k = 0
image0 = np.ones(image1.shape)*(-1)
for i in range(hierarchy.shape[1]):
    xmin, xmax = np.amin(contours[i][:, 0, 1]), np.amax(contours[i][:, 0, 1])
    ymin, ymax = np.amin(contours[i][:, 0, 0]), np.amax(contours[i][:, 0, 0])

    cv2.drawContours(image0[:xmax+1, :ymax+1],contours, i, i, -1)  # Draw only the target cell as i on a zero matrix
    cellsize = np.sum(image0[xmin:xmax+1, ymin:ymax+1] == i)
    
    # imshow colormap and hist are affected by outliers, so the range can be restricted
    if mincellsize <= cellsize <= maxcellsize:
        cellsizelist.append(cellsize)
        cv2.drawContours(image_cs[:xmax+1, :ymax+1],contours, i, int(cellsize), -1)  # Draw each cell on a zero matrix with its area value
        

print(f"{len(cellsizelist)} cells in a image.")

plt.hist(cellsizelist,bins=20)#bins=20,range=(0, 400)
plt.show()

fig=plt.figure(figsize=(20,20))
im=plt.imshow(image_cs,cmap='nipy_spectral')
fig.subplots_adjust(right=0.8)# Leave space for the color bar
cbar_ax=plt.axes([0.805,0.16,0.01,0.67])# Position of the color bar ([x, y, width, height])
plt.colorbar(im,cax=cbar_ax)
plt.show()

## Variation in the number of analyzed cells

In [ ]:
# Blue: How the number of cells stored in targetIndex changes over time
# Orange: How the number of "recognized cell-like objects" stored in mxy changes over time
plt.figure(figsize=(15,3))
plt.plot(np.sum(targetIndex!=-1,axis=1)[:T],color='b',label='targetIndex')
plt.plot([m.shape[0] for m in mxy][:T],color='orange',label='mxy')
plt.legend()
plt.show()
print("Average number of cells during observation period:", np.average(np.sum(targetIndex!=-1,axis=1)[:T]))

In [ ]:
# Where is FITTING_FRAME approximately?

fI_avg_cell=np.sum(fluoIntensity.copy(),axis=2)/np.sum(fluoIntensity!=0,axis=2)
fI_avg_cell[np.isnan(fI_avg_cell)]=0
fI_avg=np.sum(fI_avg_cell,axis=1)/np.sum(targetIndex!=-1,axis=1)
fI_max=np.sum(np.amax(fluoIntensity,axis=2),axis=1)/np.sum(targetIndex!=-1,axis=1)

plt.plot(fI_max,'b',label="max")
plt.plot(fI_avg,'r',label="average")
plt.plot(fI_max-fI_avg,'g',label="max-avg")
for add_time_sec in ADD_TIME_SEC:
    plt.vlines(add_time_sec*FPS, np.amax(fI_max[~np.isnan(fI_max)])*1.1, 0, color="black")
plt.ylim([0,2000])
plt.grid(color='black', linestyle=':', linewidth=0.3,which='both', axis="x")
plt.grid(color='black', linestyle=':', linewidth=0.3,which='both', axis="y")
plt.legend()
plt.show()
print(f"Frame where max brightness deviates most from average brightness: {np.argmax(fI_max-fI_avg)}")

In [ ]:
# Track the average change in cell length

avg_cell_length=np.sum(np.sum(fluoIntensity!=0,axis=2),axis=1)/np.sum(targetIndex!=-1,axis=1)
plt.plot(avg_cell_length)
plt.ylim(bottom=20)
plt.xlabel("frame[f]")
plt.ylabel("cellLength[pix]")
plt.show()